This cell is responsible for loading pre-computed data artifacts generated in a previous processing step, ensuring that the current analysis can proceed without needing to re-run computationally intensive tasks like API calls or clustering algorithms. By retrieving various data structures, such as embeddings, cluster labels, and summaries, it sets the stage for robustness checks that assess the sensitivity of the main findings to different methodological choices, thereby enhancing the reliability of the overall analysis in public dialogue research.

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du

OUTPUT_FOLDER     = Path("outputs")
CHECKPOINT_FOLDER = Path("checkpoints")

a = du.load_artifacts(OUTPUT_FOLDER, CHECKPOINT_FOLDER)

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

This cell imports essential functions from the `dialogue_utils` module, which are crucial for the subsequent analysis of clustering results. By facilitating the mapping of clusters to labels and lenses, as well as ensuring proper formatting for HTML output, these functions support the robustness checks aimed at evaluating how variations in embedding models, clustering parameters, and extraction prompts influence the overall findings in public dialogue research.

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

This cell generates a stratified random sample of text paragraphs for human validation, specifically focusing on concerns identified in the analysis. By ensuring representation across different technologies and preparing the data for hand-coding, it addresses methodological gaps related to inter-rater reliability, thereby enhancing the robustness of the findings related to LLM-mediated qualitative scaling. The output facilitates the computation of agreement statistics, which are crucial for assessing the reliability of the automated extractions against human judgments.

In [ ]:
# @title (v16) Export human validation sample (concerns)
# Issue 12 from the audit: the paper makes methodological claims about LLM-
# mediated qualitative scaling but does not include a human inter-rater check.
# The traceability output above gives every paragraph + extracted concerns +
# cluster + lens. This cell exports a stratified-by-technology random sample
# of paragraphs ready for hand-coding.
#
# To use: download the Excel file, fill the human columns, re-import, and
# compute agreement statistics (Cohen's kappa or simple agreement) against
# the LLM extractions.

show_status("Exporting human validation sample (concerns)...")

VALIDATION_SAMPLE_N = 250

# Stratified sample by technology, with a floor per technology so small-N
# technologies are not invisible.
per_tech_floor = 10
sample_pieces = []
for tech, group in chunks_df.groupby("technology_meta"):
    n_for_tech = max(per_tech_floor, int(round(VALIDATION_SAMPLE_N * len(group) / len(chunks_df))))
    n_for_tech = min(n_for_tech, len(group))
    sample_pieces.append(group.sample(n=n_for_tech, random_state=RANDOM_SEED))

validation_sample = pd.concat(sample_pieces, ignore_index=True)
# Trim if oversized after the floor
if len(validation_sample) > VALIDATION_SAMPLE_N * 1.5:
    validation_sample = validation_sample.sample(int(VALIDATION_SAMPLE_N * 1.5), random_state=RANDOM_SEED)

# Attach LLM extractions for each sampled chunk
sample_concerns = (concerns_df.groupby("chunk_id")["concern"]
                   .apply(lambda s: " | ".join(s))
                   .rename("llm_concerns"))
validation_sample = validation_sample.merge(sample_concerns, on="chunk_id", how="left")
validation_sample["llm_concerns"] = validation_sample["llm_concerns"].fillna("(NO_CONCERN)")
validation_sample["llm_concern_count"] = validation_sample["llm_concerns"].apply(
    lambda s: 0 if s == "(NO_CONCERN)" else s.count("|") + 1
)

# Empty columns for the human coder
for col in ["human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes"]:
    validation_sample[col] = ""

cols_for_export = [
    "chunk_id", "source_file", "technology_meta", "year", "word_count",
    "text", "llm_concerns", "llm_concern_count",
    "human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes",
]
validation_sample = validation_sample[cols_for_export]

from pub_dialogue.address import _clean_for_xlsx

validation_sample_xlsx = validation_sample.applymap(_clean_for_xlsx)
validation_sample_xlsx.to_excel(OUTPUT_FOLDER / "human_validation_sample_concerns.xlsx", index=False)
validation_sample.to_csv(OUTPUT_FOLDER / "human_validation_sample_concerns.csv", index=False)

print(f"Exported {len(validation_sample)} chunks for human validation.")
print(f"Stratified by technology; per-technology floor = {per_tech_floor}.")
print()
print("Suggested human columns:")
print("  human_any_concern: yes / no / unclear")
print("  human_concerns: pipe-separated phrases the human extracted")
print("  human_retain_llm_extraction: yes / no / partial")
print("  human_notes: free text")

show_complete("Human validation sample exported. See human_validation_sample_concerns.xlsx")

This cell imports functions related to calculating normalized entropy and AI fingerprints from the dialogue_utils module, which are essential for assessing the robustness of the analysis. By evaluating how sensitive the main findings are to different embedding models and clustering parameters, this step ensures that the conclusions drawn from the dialogue data are reliable and not artifacts of specific methodological choices.

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

This cell evaluates the stability of public concerns regarding AI by calculating the mean absolute rank change of various AI-related topics over time. By assessing the volatility in rankings, it provides insights into how perceptions of AI issues fluctuate, which is crucial for understanding the dynamics of public dialogue and the potential impact of these concerns on policy and societal attitudes. The resulting visualization and data frame highlight trends in concern stability, informing further analysis of how these perceptions may influence future discussions and decisions in the field.

In [ ]:
# @title Assess volatility of AI concern rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public concerns over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.show()

display(vol_df)


This cell imports essential functions for parsing year data and tokenizing text, which are critical for preprocessing dialogue data in the robustness checks. By ensuring that the data is properly formatted and segmented, the analysis can accurately assess how variations in embedding models, clustering parameters, and extraction prompts influence the main findings, thereby validating the robustness of the results.

In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

This cell evaluates the emergence of new concern types in AI dialogues over time by analyzing the active clusters of concerns and calculating the share of newly identified clusters each year. By visualizing this data, it highlights trends in the novelty of discourse, which is crucial for understanding how public concerns evolve in response to advancements in AI technology. This analysis contributes to the robustness checks by assessing the sensitivity of findings to the dynamics of concern types, thereby informing the overall interpretation of dialogue trends in the context of AI.

In [ ]:
# @title Assess novelty of concern types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
present = (ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing concern clusters")
plt.title("Novelty of concern-types in AI discourse over time")
plt.tight_layout()
plt.show()

display(nov2)


---
# SECTION 10: Sensitivity analysis: number of clusters
We re-run *selected headline outputs* for **k = 60** and **k = 90** (baseline **k = 75**) to assess whether substantive conclusions depend on clustering resolution.

**Notes:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline (k=75) cluster centroid (cosine similarity) and inherit baseline lens memberships. This provides a consistent lens vocabulary without re-running LLM labelling.


In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

This section of the notebook addresses the issue of data cleanliness by ensuring that any illegal control characters from PDF extraction artifacts are removed before exporting results to Excel. This step is crucial for maintaining the integrity of the data during robustness checks, as it prevents potential errors in analysis that could arise from corrupted or improperly formatted values, thereby enhancing the reliability of the sensitivity analysis related to clustering granularity.

In [ ]:
# _clean_for_xlsx is defined locally in the export cell above (Cell 94).
# It strips openpyxl-illegal control characters (PDF extraction artefacts)
# from Excel export values and is NOT imported from dialogue_utils.

This cell imports functions related to normalized entropy and AI fingerprinting, which are essential for evaluating the sensitivity of clustering outcomes based on different numbers of clusters. By analyzing how variations in clustering granularity affect the robustness of the findings, this section contributes to understanding the stability and reliability of the dialogue analysis results, ensuring that conclusions drawn are not artifacts of arbitrary model choices.

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

This cell evaluates the stability of AI benefit rankings over time by calculating the mean absolute rank change for each year based on normalized benefit data. By visualizing the volatility, it provides insights into how consistent public perceptions of AI benefits are, which is crucial for understanding the reliability of the findings and their implications for policy and public dialogue. A lower volatility indicates greater stability in the rankings, suggesting that the perceived benefits of AI are becoming more entrenched or widely accepted.

In [ ]:
# @title Assess volatility of AI benefit rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "benefit_ai_year" not in globals():
    raise NameError("benefit_ai_year not found. Run cell 80 first.")

ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public benefits over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_volatility_over_time.png", dpi=150, bbox_inches='tight')
plt.show()

display(vol_df)


This section prepares for a sensitivity analysis by importing necessary functions to parse dialogue data by year and tokenize the text. These steps are crucial for examining how variations in clustering granularity, specifically the number of clusters (k), influence the robustness of the main findings in the context of public dialogue analysis. Understanding these sensitivities helps ensure that the conclusions drawn from the data are reliable and not artifacts of specific methodological choices.

In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

This cell evaluates the novelty of benefit types discussed in AI dialogues over time by analyzing the emergence of new clusters of benefits each year. It calculates the share of newly identified clusters relative to the total active clusters, providing insights into how discussions around AI benefits evolve and potentially indicating shifts in public perception or emerging trends. This analysis is crucial for understanding the dynamics of AI discourse and assessing the robustness of findings related to benefit recognition in public dialogue.

In [ ]:
# @title Assess novelty of benefit types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

present = (benefit_ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing benefit clusters")
plt.title("Novelty of benefit-types in AI discourse over time")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_cluster_novelty.png", dpi=150, bbox_inches='tight')
plt.show()

display(nov2)


---
# SECTION 10B: Sensitivity analysis: number of clusters (benefits)
We re-run *selected headline outputs* for **k = 60** and **k = 90** (baseline **k = 75**) to assess whether substantive conclusions depend on clustering resolution.

**Notes:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline (k=75) cluster centroid (cosine similarity) and inherit baseline lens memberships. This provides a consistent lens vocabulary without re-running LLM labelling.


In [ ]:
# @title Export all analysis outputs

import os
import zipfile
from pathlib import Path
from google.colab import files

# Where outputs live
OUTPUT_DIR = Path(OUTPUT_FOLDER)
ARTIFACT_DIR = Path(globals().get("ARTIFACT_DIR", "analysis_artifacts"))

ZIP_NAME = "public_dialogue_analysis_outputs.zip"
ZIP_PATH = OUTPUT_DIR / ZIP_NAME

print("Preparing ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add OUTPUT_FOLDER contents
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and path.name != ZIP_NAME:
            zf.write(path, arcname=f"outputs/{path.relative_to(OUTPUT_DIR)}")

    # Optionally add analysis_artifacts if it exists
    if ARTIFACT_DIR.exists():
        for path in ARTIFACT_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=f"analysis_artifacts/{path.relative_to(ARTIFACT_DIR)}")

print(f"ZIP written to: {ZIP_PATH}")
print("Files included:")

for p in sorted(OUTPUT_DIR.glob("*")):
    if p.name != ZIP_NAME:
        print(" - outputs/", p.name)

if ARTIFACT_DIR.exists():
    print(" - analysis_artifacts/ (included)")

# Trigger download in Colab
files.download(str(ZIP_PATH))
